# CatBoost + Cleveland Original Data

The competition data is synthetically generated from the UCI Cleveland Heart Disease dataset.
Adding the ~297 original Cleveland rows gives the model access to the "true" source distribution.

The Cleveland data uses **identical integer codes** to the competition data — no remapping needed.
Only 6 rows are dropped (missing `number_of_vessels_fluro` / `thallium`).

**Baseline**: catboost_default_proba LB = 0.95356 | CV AUC = 0.95533

## Imports & Data

In [1]:
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import catboost as cb

KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(pd.read_csv(DATA_DIR / 'train.csv'))
test  = prep(pd.read_csv(DATA_DIR / 'test.csv'))
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

FEATURES     = [c for c in train.columns if c not in ['heart_disease', 'id']]
CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']

X_test = test[FEATURES]
print(f'Train: {train.shape}  Test: {test.shape}')

Train: (630000, 15)  Test: (270000, 14)


## Load & Align Cleveland Data

The UCI processed Cleveland file uses the same integer codes as the competition:
- sex: 0/1, chest_pain_type: 1-4, fbs_over_120: 0/1, ekg_results: 0/1/2
- exercise_angina: 0/1, slope_of_st: 1-3, number_of_vessels_fluro: 0-3, thallium: 3/6/7

Target: 0 = absent, 1-4 = present → binarize to 0/1.

In [2]:
UCI_COLS = ['age', 'sex', 'chest_pain_type', 'bp', 'cholesterol', 'fbs_over_120',
            'ekg_results', 'max_hr', 'exercise_angina', 'st_depression',
            'slope_of_st', 'number_of_vessels_fluro', 'thallium', 'heart_disease']

# Locate Cleveland file (works both locally and on Kaggle with dataset attached)
cleve_path = Path('data/uci_raw/processed.cleveland.data')
cleve = pd.read_csv(cleve_path, header=None, names=UCI_COLS, na_values='?')

# Binarize target: 0=absent, 1-4=present
cleve['heart_disease'] = (cleve['heart_disease'] > 0).astype(int)

# Drop the 6 rows with missing values (4 in vessels_fluro, 2 in thallium)
cleve = cleve.dropna().reset_index(drop=True)

# Cast to same dtypes as competition data
for col in ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']:
    cleve[col] = cleve[col].astype(float)
for col in CAT_FEATURES:
    cleve[col] = cleve[col].astype(int)
cleve['heart_disease'] = cleve['heart_disease'].astype(int)
cleve['id'] = range(-len(cleve), 0)  # negative ids to distinguish from train

print(f'Cleveland rows (after dropping NaN): {len(cleve)}')
print(f'Positive rate — Cleveland: {cleve.heart_disease.mean():.3f}  '
      f'Train: {train.heart_disease.mean():.3f}')

Cleveland rows (after dropping NaN): 297
Positive rate — Cleveland: 0.461  Train: 0.448


In [3]:
# Augment training set
train_aug = pd.concat([train, cleve[train.columns]], ignore_index=True)

X_aug = train_aug[FEATURES]
y_aug = train_aug['heart_disease'].values

print(f'Original train: {len(train):,}  +  Cleveland: {len(cleve)}  =  Augmented: {len(train_aug):,}')
print(f'Positive rate (augmented): {y_aug.mean():.4f}')

Original train: 630,000  +  Cleveland: 297  =  Augmented: 630,297
Positive rate (augmented): 0.4483


## CatBoost 5-Fold CV (Default Params + Cleveland Data)

In [4]:
BASELINE_CV  = 0.95533  # CatBoost default, no Cleveland
BASELINE_LB  = 0.95356

cv5  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof  = np.zeros(len(y_aug))
test_folds = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(cv5.split(X_aug, y_aug)):
    m = cb.CatBoostClassifier(
        iterations=1000,
        learning_rate=0.1,
        depth=6,
        task_type='GPU',
        cat_features=CAT_FEATURES,
        random_state=42,
        verbose=0,
        eval_metric='AUC',
        early_stopping_rounds=50,
    )
    m.fit(
        X_aug.iloc[tr_idx], y_aug[tr_idx],
        eval_set=(X_aug.iloc[val_idx], y_aug[val_idx]),
        verbose=False
    )
    oof[val_idx]      = m.predict_proba(X_aug.iloc[val_idx])[:, 1]
    test_folds[fold]  = m.predict_proba(X_test)[:, 1]
    fold_auc = roc_auc_score(y_aug[val_idx], oof[val_idx])
    print(f'Fold {fold+1}: AUC={fold_auc:.5f}  best_iter={m.get_best_iteration()}')

# Only evaluate OOF on the original training rows (not Cleveland)
n_orig    = len(train)
cv_auc    = roc_auc_score(y_aug[:n_orig], oof[:n_orig])
cv_auc_all = roc_auc_score(y_aug, oof)
test_preds = test_folds.mean(axis=0)

print(f'\nCV AUC (original rows only): {cv_auc:.5f}  ({cv_auc - BASELINE_CV:+.5f} vs baseline)')
print(f'CV AUC (all rows incl. Cleve): {cv_auc_all:.5f}')

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1: AUC=0.95492  best_iter=589


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2: AUC=0.95553  best_iter=559


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 3: AUC=0.95500  best_iter=625


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4: AUC=0.95574  best_iter=472


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5: AUC=0.95547  best_iter=605

CV AUC (original rows only): 0.95535  (+0.00002 vs baseline)
CV AUC (all rows incl. Cleve): 0.95533


## Submit

In [5]:
label = 'cat_cleveland_aug'
fname = f'submissions/{label}.csv'

sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')

desc = f'{label} | cv_acc=N/A | cv_auc={cv_auc:.4f}'
print(f'desc: {desc}')

r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

Saved: submissions/cat_cleveland_aug.csv
desc: cat_cleveland_aug | cv_acc=N/A | cv_auc=0.9553
cat_cleveland_aug: submitted
